In [1]:
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.exceptions import OutputParserException
from llm_helper import llm

c:\Users\saite\OneDrive\Desktop\Linkedin Post generator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1107.44it/s]
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [2]:
def extract_metadata(post):
    
    template = '''
        
            You are a JSON generator.

            Your task is to extract metadata from a LinkedIn post.

            You MUST return ONLY valid JSON.
            If you add any extra text, the answer is WRONG.

            Rules:
            1. Output must be ONLY JSON. No explanations, no comments, no markdown, no words before or after JSON.
            2. No explanations.
            3. No markdown.
            4. No words before or after JSON.
            5. JSON must contain EXACTLY these keys:
            - line_count
            - language
            - tags
            6. tags must contain MAXIMUM 2 items.
            7. language must be either "English" or "Hinglish".

            Now extract metadata from the below post in the below mentioned example format only without any additional information:

            {post}
            
            
            JSON:
            '''
    
    pt = PromptTemplate.from_template(template)
    chain = pt | llm 
    response = chain.invoke({'post': post})
    
    
    try:
        json_parser = JsonOutputParser()
        res = json_parser.parse(response)
    except OutputParserException:
        raise OutputParserException('Invalid JSON returned by model')
    return res

In [3]:
def process_posts(raw_file_path="data/raw_posts.json", processed_file_path="data/processed_posts.json"):
    with open(raw_file_path, encoding='utf-8') as file:
        posts = json.load(file)
        enriched_posts = []
        for post in posts:
            metadata = extract_metadata(post['text'])
            post_with_metadata = post | metadata
            enriched_posts.append(post_with_metadata)
        
        for epost in enriched_posts:
            print(epost)
    

In [5]:
process_posts(raw_file_path="sample_posts.json", processed_file_path="data/processed_posts.json")

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

{'text': "Just saw a LinkedIn Influencer with 'Organic Growth' written in the profile with 65K+ followers claiming that he can help you in growing your platform, copying the posts from other influencers.", 'engagement': 90, 'line_count': 1, 'language': 'English', 'tags': [{'name': 'Organic Growth', 'count': 65000}, {'name': 'Influencer', 'count': 65000}]}
{'text': "Jobseekers, this one’s for you.\n Every application, every interview, every follow-up… the pressure is immense.\n And I know what you're thinking: Am I not good enough? \n But let me tell you, this isn’t about you or your skills. It’s about a broken system where 60% of applicants never hear back. \n Your mental health is not worth sacrificing for a system that doesn’t acknowledge your worth. \n Please remember, taking care of yourself is the real priority. \n Your dream job will come, but for now, breathe. 🌻", 'engagement': 347, 'line_count': 1, 'language': 'English', 'tags': [{'key': 'Hinglish', 'value': 'Yes'}, {'key': 'Jo

In [ ]:
import json
import pandas as pd

In [ ]:
class FewShotExample:
    def __init__(self, file_path="processed_posts.json"):
        self.df = None
        self.unique_tags = None
        self.load_posts(file_path)
        
    def load_posts(self, file_path):
        with open(file_path, encoding='utf-8') as file:
            posts = json.load(file)
            df = pd.json_normalize(posts)
            self.df = df

In [ ]:
FewShotExample(file_path="processed_posts.json")